# Lab 3: Logistics, LDA, QDA
**Student Name**: Amelia Nguyen

**Date**: February 17, 2026

In [72]:
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots
import statsmodels.api as sm
from ISLP import load_data
from ISLP.models import ModelSpec as MS
from ISLP.models import *
from sklearn.model_selection import *
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis as QDA
from sklearn.metrics import confusion_matrix 
from sklearn.linear_model import LogisticRegression
from functools import partial
from sklearn.base import clone

### Question 1: 5 multiple choice questions
Completed

### Question 2: Sales information for Citrus Hill and Minute Maid orange juice (OJ data in ISLR2 package).


In [3]:
OJ = load_data("OJ")

In [4]:
OJ.head(5)

,Purchase,WeekofPurchase,StoreID,PriceCH,PriceMM,DiscCH,DiscMM,SpecialCH,SpecialMM,LoyalCH,SalePriceMM,SalePriceCH,PriceDiff,Store7,PctDiscMM,PctDiscCH,ListPriceDiff,STORE
0,CH,237,1,1.75,1.99,0.00,0.0,0,0,0.500000,1.99,1.75,0.24,No,0.000000,0.000000,0.24,1
1,CH,239,1,1.75,1.99,0.00,0.3,0,1,0.600000,1.69,1.75,-0.06,No,0.150754,0.000000,0.24,1
2,CH,245,1,1.86,2.09,0.17,0.0,0,0,0.680000,2.09,1.69,0.40,No,0.000000,0.091398,0.23,1
3,MM,227,1,1.69,1.69,0.00,0.0,0,0,0.400000,1.69,1.69,0.00,No,0.000000,0.000000,0.00,1
4,CH,228,7,1.69,1.69,0.00,0.0,0,0,0.956535,1.69,1.69,0.00,Yes,0.000000,0.000000,0.00,0


In [5]:
OJ.describe()

,WeekofPurchase,StoreID,PriceCH,PriceMM,DiscCH,DiscMM,SpecialCH,SpecialMM,LoyalCH,SalePriceMM,SalePriceCH,PriceDiff,PctDiscMM,PctDiscCH,ListPriceDiff,STORE
count,1070.000000,1070.000000,1070.000000,1070.000000,1070.000000,1070.000000,1070.000000,1070.000000,1070.000000,1070.000000,1070.000000,1070.000000,1070.000000,1070.000000,1070.000000,1070.000000
mean,254.381308,3.959813,1.867421,2.085411,0.051860,0.123364,0.147664,0.161682,0.565782,1.962047,1.815561,0.146486,0.059298,0.027314,0.217991,1.630841
std,15.558286,2.308984,0.101970,0.134386,0.117474,0.213834,0.354932,0.368331,0.307843,0.252697,0.143384,0.271563,0.101760,0.062232,0.107535,1.430387
min,227.000000,1.000000,1.690000,1.690000,0.000000,0.000000,0.000000,0.000000,0.000011,1.190000,1.390000,-0.670000,0.000000,0.000000,0.000000,0.000000
25%,240.000000,2.000000,1.790000,1.990000,0.000000,0.000000,0.000000,0.000000,0.325257,1.690000,1.750000,0.000000,0.000000,0.000000,0.140000,0.000000
50%,257.000000,3.000000,1.860000,2.090000,0.000000,0.000000,0.000000,0.000000,0.600000,2.090000,1.860000,0.230000,0.000000,0.000000,0.240000,2.000000
75%,268.000000,7.000000,1.990000,2.180000,0.000000,0.230000,0.000000,0.000000,0.850873,2.130000,1.890000,0.320000,0.112676,0.000000,0.300000,3.000000
max,278.000000,7.000000,2.090000,2.290000,0.500000,0.800000,1.000000,1.000000,0.999947,2.290000,2.090000,0.640000,0.402010,0.252688,0.440000,4.000000


#### (a) Purchase is the response and other variables are predictors. Split the data into a training set (80%) and a test set (20%).

In [6]:
X = OJ.drop(columns='Purchase')
y = OJ['Purchase']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify = y)

In [7]:
print(len(X_test))
print(len(X_test) / len(OJ))

214
0.2


#### (b) Perform LDA on the training data in order to predict Purchase using the threshold = 0.5 where CH = 1, MM=0. What is the test error?

In [8]:
lda = LDA()

In [9]:
# transform y to binary (CH = 1, MM=0)
y_train_new = (y_train == 'CH').astype(int)
y_test_new = (y_test == 'CH').astype(int)

In [10]:
X_train

,WeekofPurchase,StoreID,PriceCH,PriceMM,DiscCH,DiscMM,SpecialCH,SpecialMM,LoyalCH,SalePriceMM,SalePriceCH,PriceDiff,Store7,PctDiscMM,PctDiscCH,ListPriceDiff,STORE
251,270,7,1.86,2.13,0.27,0.00,1,0,0.990137,2.13,1.59,0.54,Yes,0.000000,0.145161,0.27,0
372,274,7,1.86,2.13,0.47,0.54,1,0,0.314286,1.59,1.39,0.20,Yes,0.253521,0.252688,0.27,0
907,238,1,1.75,1.99,0.00,0.00,0,0,0.163840,1.99,1.75,0.24,No,0.000000,0.000000,0.24,1
419,259,3,1.99,2.29,0.00,0.00,0,0,0.047628,2.29,1.99,0.30,No,0.000000,0.000000,0.30,3
280,243,3,1.99,2.23,0.00,0.00,0,0,0.005765,2.23,1.99,0.24,No,0.000000,0.000000,0.24,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35,256,3,1.99,2.29,0.00,0.00,0,0,0.635200,2.29,1.99,0.30,No,0.000000,0.000000,0.30,3
418,258,2,1.86,2.18,0.00,0.00,0,0,0.059535,2.18,1.86,0.32,No,0.000000,0.000000,0.32,2
294,275,2,1.96,2.18,0.00,0.80,0,1,0.000254,1.38,1.96,-0.58,No,0.366972,0.000000,0.22,2
73,278,4,2.09,2.09,0.20,0.00,0,0,0.855775,2.09,1.89,0.20,No,0.000000,0.095694,0.00,4


In [11]:
# other transformation
X_train['Store7'] = (X_train['Store7'] == 'Yes').astype(int)
X_test['Store7'] = (X_test['Store7'] == 'Yes').astype(int)

In [12]:
y_train_new.unique()

array([1, 0])

In [13]:
lda.fit(X_train, y_train_new)

,solver,'svd'
,shrinkage,None
,priors,None
,n_components,None
,store_covariance,False
,tol,0.0001
,covariance_estimator,None


In [14]:
lda_pred = lda.predict(X_test)
cf = confusion_matrix(lda_pred, y_test_new)
cf

array([[ 62,  12],
       [ 21, 119]])

In [15]:
confusion_table = pd.DataFrame(cf, columns=["True MM", "True CH"], index =["Pred MM", "Pred CH"])

In [16]:
confusion_table

,True MM,True CH
Pred MM,62,12
Pred CH,21,119


In [17]:
test_error = (cf[0, 1] + cf[1, 0])/ cf.sum()
print(f"Test error of LDA: {round(test_error, 3)}")

Test error of LDA: 0.154


#### (c) Perform QDA on the training data which predicts Purchase using WeekofPurchase, PriceDiff and ListPriceDiff using the threshold = 0.5 where CH = 1, MM=0. What is the test error? 

In [18]:
X_train_c = X_train[['WeekofPurchase', 'PriceDiff', 'ListPriceDiff']]
X_test_c = X_test[['WeekofPurchase', 'PriceDiff', 'ListPriceDiff']]

In [19]:
qda = QDA()
qda.fit(X_train_c, y_train_new)

,priors,None
,reg_param,0.0
,store_covariance,False
,tol,0.0001


In [20]:
qda_pred = qda.predict(X_test_c)
cf_c = confusion_matrix(qda_pred, y_test_new)
cf_c

array([[ 33,  22],
       [ 50, 109]])

In [21]:
confusion_table_c = pd.DataFrame(cf, columns=["True MM", "True CH"], index =["Pred MM", "Pred CH"])
confusion_table_c

,True MM,True CH
Pred MM,62,12
Pred CH,21,119


In [22]:
test_error_c = (cf_c[0, 1] + cf_c[1, 0])/ cf_c.sum()
print(f"Test error of QDA: {round(test_error_c, 3)}")

Test error of QDA: 0.336


### Question 3: Continue OJ data analysis

#### (a). Fit a logistic regression model that predicts Purchase using WeekofPurchase, PriceDiff, and ListPriceDiff, and compute LOOCV test error estimate.

In [23]:
X = OJ[['WeekofPurchase', 'PriceDiff', 'ListPriceDiff']]
y = OJ['Purchase']
y = (y == 'CH').astype(int)

In [24]:
LogR = LogisticRegression()
LOOCV = LeaveOneOut()

In [25]:
loocv_score = cross_val_score(LogR, X, y, cv = LOOCV, scoring ="accuracy")

In [26]:
loocv_error = 1 - loocv_score.mean()
print(f"LOOCV test error estimate: {round(loocv_error,3)}")

LOOCV test error estimate: 0.355


#### (b). Fit a logistic regression model that predicts Purchase using WeekofPurchase, PriceDiff, and ListPriceDiff, and compute 5 fold CV test error estimate.

In [27]:
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_val_score(LogR, X, y, cv = kfold, scoring = "accuracy")
cv_error = 1 - cv_results.mean()
print(f"5-fold CV test error estimate: {round(cv_error,3)}")

5-fold CV test error estimate: 0.353


#### (c). Fit multiple logistic regression models
    M1: Predict Purchase using WeekofPurchase
    M2: Predict Purchase using WeekofPurchase and PriceDiff
    M3: Predict Purchase using WeekofPurchase, PriceDiff and ListPriceDiff
Compare the above 3 models 10 fold CV test errors, and select the optimal model (model
selection via Cross Validation procedure).

In [42]:
kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=10)

In [46]:
# Model 1
model1_X = OJ[['WeekofPurchase']]
model1_y = OJ['Purchase']
model1_y = (model1_y == 'CH').astype(int)
cv_results = cross_val_score(LogR, model1_X, model1_y, cv = kfold, scoring = "accuracy")
cv_error = 1 - cv_results.mean()
print(f"10-fold CV test error estimate for M1: {round(cv_error, 4)}")

10-fold CV test error estimate for M1: 0.3879


In [47]:
# Model 2
model2_X = OJ[['WeekofPurchase', 'PriceDiff']]
model2_y = OJ['Purchase']
model2_y = (model2_y == 'CH').astype(int)
cv_results = cross_val_score(LogR, model2_X, model2_y, cv = kfold, scoring = "accuracy")
cv_error = 1 - cv_results.mean()
print(f"10-fold CV test error estimate for M1: {round(cv_error, 4)}")

10-fold CV test error estimate for M1: 0.3523


In [48]:
# Model 3
model3_X = OJ[['WeekofPurchase', 'PriceDiff', 'ListPriceDiff']]
model3_y = OJ['Purchase']
model3_y = (model3_y == 'CH').astype(int)
cv_results = cross_val_score(LogR, model3_X, model3_y, cv = kfold, scoring = "accuracy")
cv_error = 1 - cv_results.mean()
print(f"10-fold CV test error estimate for M1: {round(cv_error, 4)}")

10-fold CV test error estimate for M1: 0.3542


We can see that Model 2 is the optimal with the lowest test-error when applying 10 fold CV.

### Question 4: Continue OJ data analysis

#### (a) Using glm(), determine the estimated standard errors for thecoefficient associated with WeekofPurchase and PriceDiff in a multiple logistic regression model that uses both predictors.

In [50]:
OJ = load_data("OJ")
X = OJ[['WeekofPurchase', 'PriceDiff']]
X = sm.add_constant(X)
y = (OJ['Purchase']=='CH').astype(int)

model = sm.GLM(y, X, family = sm.families.Binomial())
result = model.fit()

In [51]:
print(result.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:               Purchase   No. Observations:                 1070
Model:                            GLM   Df Residuals:                     1067
Model Family:                Binomial   Df Model:                            2
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -664.39
Date:                Wed, 18 Feb 2026   Deviance:                       1328.8
Time:                        12:52:33   Pearson chi2:                 1.06e+03
No. Iterations:                     4   Pseudo R-squ. (CS):            0.09099
Covariance Type:            nonrobust                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const             -4.5950      1.072     -4.

* SE of WeekofPurchase = 0.004
* SE of PriceDiff = 0.253

#### (b) Write a function, boot.fn(), that takes as input the OJ data set as well as an index of the observations, and that outputs the coefficient estimates for WeekofPurchase and PriceDiff in the multiple logistic regression model.

In [96]:
def boot_fn(model_matrix, D, idx):
    D_ = D.loc[idx]
    Y_ = (D_['Purchase'] == 'CH').astype(int)
    X_ = clone(model_matrix).fit_transform(D_)
    model = sm.GLM(Y_, X_, family=sm.families.Binomial())
    result = model.fit()
    return result.params[1:]
    
hp_func = partial(boot_fn, MS(['WeekofPurchase', 'PriceDiff']))
rng = np.random.default_rng(0)

boot_estimates = np.array([hp_func(OJ, 
                                   rng.choice(OJ.index, len(OJ), 
                                    replace=True)) for _ in range(10)])

In [98]:
estimates = pd.DataFrame(boot_estimates, columns=['WeekofPurchase', 'PriceDiff'])

In [99]:
estimates

,WeekofPurchase,PriceDiff
0,0.019035,1.801574
1,0.013928,2.450037
2,0.025510,2.512429
3,0.020266,2.305690
4,0.020111,2.463993
5,0.018651,2.676879
6,0.012380,2.126163
7,0.018723,2.169012
8,0.013945,2.206582
9,0.015732,2.255475


#### (c) Use the boot() function together with your boot.fn() function to estimate the standard errors of the logistic regression coefficients for WeekofPurchase and PriceDiff.

In python, I'm creating a boot_SE() function to estimate the standard error of the coefficients

In [100]:
def boot_SE(func,
            D,
            n=None,
            B=1000, #1000 bootstrap reps
            seed=0):
    rng = np.random.default_rng(seed)
    first_, second_ = 0, 0
    n = n or D.shape[0]
    for _ in range(B):
        idx = rng.choice(D.index,
                         n,
                         replace=True)
        value = func(D, idx)
        first_ += value
        second_ += value**2
    return np.sqrt(second_ / B - (first_ / B)**2)

In [101]:
hp_se = boot_SE(hp_func, OJ, B=1000, seed=10)
hp_se

WeekofPurchase    0.004311
PriceDiff         0.248216
dtype: float64

This estimates the bootstrap estimate for:
* SE(WeekofPurchase) = 0.004311
* SE(PriceDiff) = 0.248216.

#### (d) Comment on the estimated standard errors obtained using the glm() function and using your bootstrap function

GLM() result:
* SE of WeekofPurchase = 0.004
* SE of PriceDiff = 0.253
  
Bootstrap result:
* SE(WeekofPurchase) = 0.004311
* SE(PriceDiff) = 0.248216.

Coefficient Standard Error using Bootstrap is greatly close to fitting multiple logistics regression for both predictors. This suggests that the model SE and logistics regression assumptions are quite reliable.